In [ ]:
# Cell 1 : Importing packages 
%matplotlib widget
import os
import math
import copy
import time
import random
from pathlib import Path
from typing import List, Dict

import numpy as np
from PIL import ImageFile

import torch
import torch.nn as nn
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import roc_auc_score

import matplotlib.pyplot as plt
from arvo import NotebookTrainingDashboard

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Resized,
    ScaleIntensityd,
    RandFlipd,
    RandRotated,
    RandZoomd,
    RandGaussianNoised,
    EnsureTyped,
    MapTransform,
)
from monai.data import Dataset, DataLoader
from monai.utils import set_determinism
from monai.visualize import OcclusionSensitivity

from transformers import AutoConfig, AutoImageProcessor, AutoModel
from dataclasses import dataclass

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()

In [ ]:
# Cell 2 : Configuration

@dataclass
class Config:
    data_root: str = "data"
    model_name: str = "facebook/dinov3-vits16-pretrain-lvd1689m"

    image_size: int = 224
    batch_size: int = 32
    num_workers: int = 4
    
    use_pretrained: bool = True   # False => 1-Stage scratch training. True => 2-Stage probe/finetune.

    use_weighted_sampler: bool = False
    use_class_weighted_loss: bool = True

    use_amp: bool = True
    grad_accum_steps: int = 1
    max_grad_norm: float = 1.0

    dashboard_update_freq: int = 20
    
    # CHANGEME 
    # Multi-stage training epochs
    probe_epochs: int = 4         # Stage 1 (Pretrained only)
    finetune_epochs: int = 4     # Stage 2 (Pretrained only)
    scratch_epochs: int = 4       # Used if use_pretrained = False

    lr: float = 1e-3              # for linear probe
    finetune_lr: float = 5e-6     # for full finetune
    scratch_lr: float = 3e-4      # for random init training
    weight_decay: float = 1e-4

    dropout: float = 0.2
    label_smoothing: float = 0.0
    seed: int = 42
    


CFG = Config()

In [ ]:
# Cell 3: Helper methods

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_determinism(seed=seed)


def list_classes(split_dir: Path) -> List[str]:
    return sorted([p.name for p in split_dir.iterdir() if p.is_dir()])


def build_items(split_dir: Path, class_to_idx: Dict[str, int]) -> List[Dict]:
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    items = []
    for cls_name, cls_idx in class_to_idx.items():
        cls_dir = split_dir / cls_name
        if not cls_dir.exists():
            continue
        for f in cls_dir.rglob("*"):
            if f.is_file() and f.suffix.lower() in exts:
                items.append({"image": str(f), "label": cls_idx})
    return items


def compute_class_weights(items: List[Dict], num_classes: int) -> torch.Tensor:
    counts = np.zeros(num_classes, dtype=np.int64)
    for item in items:
        counts[item["label"]] += 1
    counts = np.maximum(counts, 1)
    weights = counts.sum() / counts
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def make_weighted_sampler(items: List[Dict], num_classes: int) -> WeightedRandomSampler:
    class_weights = compute_class_weights(items, num_classes).numpy()
    sample_weights = np.array([class_weights[x["label"]] for x in items], dtype=np.float64)
    return WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )


class ProcessorNormalizeAndRenameD(MapTransform):
    """
    Final MONAI map transform:
    - ensures float32 tensor
    - applies HF processor mean/std normalization
    - renames 'image' -> 'pixel_values'
    - ensures label is torch.long
    """

    def __init__(self, keys, mean, std):
        super().__init__(keys)
        self.mean = torch.tensor(mean, dtype=torch.float32).view(-1, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(-1, 1, 1)

    def __call__(self, data):
        d = dict(data)

        img = d["image"]
        if not torch.is_tensor(img):
            img = torch.as_tensor(img, dtype=torch.float32)
        else:
            img = img.float()

        # Ensure 3 channels if something odd comes through
        if img.ndim == 2:
            img = img.unsqueeze(0).repeat(3, 1, 1)
        elif img.ndim == 3 and img.shape[0] == 1:
            img = img.repeat(3, 1, 1)

        img = (img - self.mean) / self.std

        d["pixel_values"] = img
        #d["label"] = torch.tensor(d["label"], dtype=torch.long)
        d["label"] = d["label"].long()

        return {"pixel_values": d["pixel_values"], "label": d["label"]}


def build_monai_transforms(processor: AutoImageProcessor, image_size: int, train: bool = False):
    mean = getattr(processor, "image_mean", [0.485, 0.456, 0.406])
    std = getattr(processor, "image_std", [0.229, 0.224, 0.225])

    xforms = [
        LoadImaged(keys=["image"], reader="PILReader"),
        EnsureChannelFirstd(keys=["image"]),
        Resized(keys=["image"], spatial_size=(image_size, image_size), mode="bilinear"),
        ScaleIntensityd(keys=["image"]),
    ]

    if train:
        xforms.extend(
            [
                RandFlipd(keys=["image"], prob=0.5, spatial_axis=1),
                RandRotated(
                    keys=["image"],
                    range_x=0.0,
                    range_y=0.0,
                    range_z=np.pi / 36,
                    prob=0.15,
                    keep_size=True,
                    mode="bilinear",
                    padding_mode="border",
                ),
                RandZoomd(
                    keys=["image"],
                    min_zoom=0.97,
                    max_zoom=1.03,
                    prob=0.15,
                    keep_size=True,
                    mode="bilinear",
                    padding_mode="edge",
                ),
                RandGaussianNoised(keys=["image"], prob=0.05, std=0.01),
            ]
        )

    xforms.extend(
        [
            EnsureTyped(keys=["image", "label"]),
            ProcessorNormalizeAndRenameD(keys=["image"], mean=mean, std=std),
        ]
    )

    return Compose(xforms)






In [ ]:
# Cell 4: Model Definitons

class DinoV3Classifier(nn.Module):
    def __init__(self, model_name: str, num_classes: int, dropout: float = 0.2, use_pretrained: bool = True):
        super().__init__()
        if use_pretrained:
            self.backbone = AutoModel.from_pretrained(model_name)
        else:
            config = AutoConfig.from_pretrained(model_name)
            self.backbone = AutoModel.from_config(config)

        hidden_size = getattr(
            self.backbone.config,
            "hidden_size",
            getattr(self.backbone.config, "projection_dim", None),
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, num_classes))

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        outputs = self.backbone(pixel_values=pixel_values)
        return self.head(outputs.pooler_output)


def freeze_all_backbone(model: DinoV3Classifier) -> None:
    for p in model.backbone.parameters():
        p.requires_grad = False


def unfreeze_all_backbone(model: DinoV3Classifier) -> None:
    for p in model.backbone.parameters():
        p.requires_grad = True



In [ ]:
# Cell 5: Training and evaluation logic

@torch.no_grad()
def evaluate(model, loader, device, criterion):
    model.eval()
    all_labels, all_probs, total, correct, running_loss = [], [], 0, 0, 0.0

    for batch in loader:
        x = batch["pixel_values"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * y.size(0)
        total += y.size(0)
        correct += (preds == y).sum().item()
        all_labels.append(y.cpu())
        all_probs.append(probs.cpu())

    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()

    num_classes = all_probs.shape[1]

    try:
        if num_classes == 2:
            auc = roc_auc_score(all_labels, all_probs[:, 1])
        else:
            auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")

    return {
        "loss": running_loss / max(total, 1),
        "acc": correct / max(total, 1),
        "auc": auc,
    }


def train_engine(model, train_loader, val_loader, optimizer, criterion, scaler, device, epochs, dashboard, start_epoch_offset, phase_name):
    best_val, best_state = -math.inf, None
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(1, epochs + 1):
        model.train()
        total, correct, running_loss = 0, 0, 0.0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            x = batch["pixel_values"].to(device, non_blocking=True)
            y = batch["label"].to(device, non_blocking=True)

            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=(scaler is not None)):
                logits = model(x)
                loss = criterion(logits, y)

            if scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                optimizer.step()

            optimizer.zero_grad()

            with torch.no_grad():
                probs = torch.softmax(logits, dim=1)
                preds = torch.argmax(probs, dim=1)
                total += y.size(0)
                correct += (preds == y).sum().item()
                running_loss += loss.item() * y.size(0)

            if (step + 1) % CFG.dashboard_update_freq == 0 or (step + 1) == len(train_loader):
                dashboard.update_train(
                    x=(start_epoch_offset + epoch - 1) + ((step + 1) / len(train_loader)),
                    train_loss=running_loss / total,
                    train_acc=correct / total,
                    lr=optimizer.param_groups[0]["lr"],
                    device=device,
                    epoch_label=epoch,
                    num_epochs=epochs,
                    phase=phase_name,
                )

        val_metrics = evaluate(model, val_loader, device, criterion)
        dashboard.update_val(
            epoch=start_epoch_offset + epoch,
            train_loss=running_loss / total,
            val_loss=val_metrics["loss"],
            train_acc=correct / total,
            val_acc=val_metrics["acc"],
            val_auc=val_metrics["auc"],
            lr=optimizer.param_groups[0]["lr"],
            device=device,
            num_epochs=epochs,
            phase=phase_name,
        )

        val_key = val_metrics["auc"] if not np.isnan(val_metrics["auc"]) else val_metrics["acc"]
        if val_key > best_val:
            best_val = val_key
            best_state = copy.deepcopy(model.state_dict())

        scheduler.step()

    return best_state, start_epoch_offset + epochs




In [ ]:
# Cell 6: Main method that drives all the code

def main(mode="fine_tuning"):
    cfg = CFG
    seed_everything(cfg.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    train_dir = Path(cfg.data_root) / "train"
    val_dir = Path(cfg.data_root) / "val"

    classes = list_classes(train_dir)
    class_to_idx = {c: i for i, c in enumerate(classes)}

    train_items = build_items(train_dir, class_to_idx)
    val_items = build_items(val_dir, class_to_idx)

    processor = AutoImageProcessor.from_pretrained(cfg.model_name)

    train_tf = build_monai_transforms(processor, cfg.image_size, train=True)
    eval_tf = build_monai_transforms(processor, cfg.image_size, train=False)

    train_ds = Dataset(data=train_items, transform=train_tf)
    val_ds = Dataset(data=val_items, transform=eval_tf)

    train_sampler = None
    train_shuffle = True
    if cfg.use_weighted_sampler:
        train_sampler = make_weighted_sampler(train_items, len(classes))
        train_shuffle = False

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=train_shuffle,
        sampler=train_sampler,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(cfg.num_workers > 0),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(cfg.num_workers > 0),
    )

    model = DinoV3Classifier(
        cfg.model_name,
        len(classes),
        dropout=cfg.dropout,
        use_pretrained=cfg.use_pretrained,
    ).to(device)

    class_weights = None
    if cfg.use_class_weighted_loss:
        class_weights = compute_class_weights(train_items, len(classes)).to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=cfg.label_smoothing,
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.use_amp and device.type == "cuda"))
    dashboard = NotebookTrainingDashboard()

    best_overall_state = None

    if mode == "fine_tuning":
        unfreeze_all_backbone(model)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg.finetune_lr,
            weight_decay=cfg.weight_decay,
        )
        best_overall_state, _ = train_engine(
            model,
            train_loader,
            val_loader,
            optimizer,
            criterion,
            scaler,
            device,
            cfg.finetune_epochs,
            dashboard,
            0,
            "Fine-Tuning",
        )
    elif mode == "linear_probing":
        # STAGE 1: Linear Probing
        freeze_all_backbone(model)
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=cfg.lr,
            weight_decay=cfg.weight_decay,
        )
        best_overall_state, current_epoch_offset = train_engine(
            model,
            train_loader,
            val_loader,
            optimizer,
            criterion,
            scaler,
            device,
            cfg.probe_epochs,
            dashboard,
            0,
            "Linear Probing",
        )

        model.load_state_dict(best_overall_state)
    elif mode == "random":
        unfreeze_all_backbone(model)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg.scratch_lr,
            weight_decay=cfg.weight_decay,
        )
        best_overall_state, _ = train_engine(
            model,
            train_loader,
            val_loader,
            optimizer,
            criterion,
            scaler,
            device,
            cfg.scratch_epochs,
            dashboard,
            0,
            "Scratch Training",
        )

    model.load_state_dict(best_overall_state)
    return model, class_to_idx, criterion

In [ ]:
# Cell 7: Training from Random Initialization

m, _, _ = main(mode="random")

In [ ]:
# Cell 8: Training from Foundation Model with Linear Probing

m, _, _ = main(mode="linear_probing")

In [ ]:
# Cell 9: Training from Foundation Model with Fine Tuning

m, class_to_idx, criterion = main(mode="fine_tuning")

In [ ]:
# Cell 10: Evaluation on test set with fine tuned model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
processor = AutoImageProcessor.from_pretrained(CFG.model_name)
test_dir = Path(CFG.data_root) / "test"
test_items = build_items(test_dir, class_to_idx)
eval_tf = build_monai_transforms(processor, CFG.image_size, train=False)
test_ds = Dataset(data=test_items, transform=eval_tf)

test_loader = DataLoader(
    test_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(CFG.num_workers > 0),
)

from sklearn.metrics import confusion_matrix

@torch.no_grad()
def get_preds_and_labels(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    for batch in loader:
        x = batch["pixel_values"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)

        logits = model(x)
        preds = torch.argmax(logits, dim=1)

        all_preds.append(preds.cpu())
        all_labels.append(y.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    return all_preds, all_labels
    
test_metrics = evaluate(m, test_loader, device, criterion)
preds, labels = get_preds_and_labels(m, test_loader, device)

cm = confusion_matrix(labels, preds)
print(cm)
print(test_metrics)
print(class_to_idx)

In [ ]:
# Cell 11: Method for testing with real image samples and occlusion sensitivity

def predict_single_image(
    image_path: str,
    model,
    class_to_idx, 
    device = "cuda",
    image_size: int = 224,
):
    from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd, Resized, ScaleIntensityd, EnsureTyped, Transposed
    processor = AutoImageProcessor.from_pretrained(CFG.model_name)

    model.eval()

    # Build SAME eval transforms
    eval_tf = Compose([
        LoadImaged(keys=["image"], reader="PILReader"),
        EnsureChannelFirstd(keys=["image"]),
        Resized(keys=["image"], spatial_size=(image_size, image_size), mode="bilinear"),
        Transposed(keys=["image"], indices=(0, 2, 1)),
        ScaleIntensityd(keys=["image"]),
        EnsureTyped(keys=["image", "label"]),
        ProcessorNormalizeAndRenameD(
            keys=["image"],
            mean=getattr(processor, "image_mean", [0.485, 0.456, 0.406]),
            std=getattr(processor, "image_std", [0.229, 0.224, 0.225]),
        ),
    ])

    # Dummy label (required for transform pipeline)
    data = {"image": image_path, "label": 0}

    batch = eval_tf(data)

    x = batch["pixel_values"].unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)

    probs_np = probs.cpu().numpy()[0]
    pred_idx = int(np.argmax(probs_np))
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    #return { "pred_class": idx_to_class[pred_idx], "pred_idx": pred_idx, "probs": probs_np }

    occ = OcclusionSensitivity(nn_module=model, mask_size=16)
    occ_map, _ = occ(x=x)

    heatmap = occ_map[0, pred_idx].detach().cpu().numpy()

    img = x[0].detach().cpu().permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title("Input")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap, cmap="jet")
    plt.title("Occlusion map")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(img)
    plt.imshow(heatmap, cmap="jet", alpha=0.5)
    plt.title(f"{idx_to_class[pred_idx]} ({probs_np[pred_idx]:.4f})")
    plt.axis("off")

    plt.tight_layout()
    plt.show()
    return

In [ ]:
# Cell 12: Testing on a DR positive image

predict_single_image("test_images/dr.jpg", m, class_to_idx)

In [ ]:
# Cell 13: Testing on a normal image

predict_single_image("test_images/normal.jpg", m, class_to_idx)